In [57]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np
import re

In [58]:
drug_mapping = pd.read_csv('../../data/vocab/drug-mappings.tsv', sep='\t')
drug_mapping.head()

,drugbankId,name,ttd_id,pubchem_cid,cas_num,chembl_id,zinc_id,chebi_id,kegg_cid,kegg_id,bindingDB_id,UMLS_cuis,stitch_id
0,DB13088,AZD-0424,D0QG8F,9893171.0,692054-06-1,CHEMBL3545177,NaN,NaN,NaN,NaN,NaN,C4519307,NaN
1,DB13089,Enoxolone,D06EWG,10114.0,471-53-4,CHEMBL230006,ZINC000019203131,30853,C02283,NaN,50233538.0,C0017986,NaN
2,DB13082,Nefiracetam,D0KD5P,71157.0,77191-36-7,CHEMBL260829,ZINC000000003788,135004,NaN,NaN,NaN,C0165264,NaN
3,DB13083,Talarozole,D0AN7B,9799888.0,201410-53-9,CHEMBL459505,NaN,102167,NaN,D09385,50253810.0,C2606129,NaN
4,DB13080,Roluperidone,D0SQ1W,9799284.0,359625-79-9,NaN,NaN,NaN,NaN,NaN,NaN,C4730997,NaN


In [59]:
def get_kegg_pathway(text: str) -> list[str]:
    ids = []
    start = False

    for line in text.splitlines():
        if start and re.match(r'^\s*[A-Z]{2,}\b', line):
            break

        if re.match(r'^\s*PATHWAY\b', line):
            start = True
            # e.g. "    PATHWAY     hsa04080(1137+1141)  Neuroactive..."
            for m in re.finditer(r'(\w+\d+)\(', line):
                ids.append(m.group(1))
            continue

        if start:
            m = re.match(r'^\s*(\w+\d+)\(', line)
            if m:
                ids.append(m.group(1))

    return ids

In [28]:
data = []

for idx, row in drug_mapping.iterrows():
    print(f'{idx}/{len(drug_mapping)}', end='\r')
    drugbank_id = row['drugbankId']
    kegg_id = row['kegg_id']
    if kegg_id is not np.nan:
        r = requests.get(f'https://rest.kegg.jp/get/dr:D09385')
        if r.status_code == 200:
            data.append({
                'kegg_id': kegg_id,
                'drugbank_id': drugbank_id,
                'data': r.text
            })
        else:
            print(f'{kegg_id} not found')

df = pd.DataFrame(data)
df = df.dropna()
df = df.drop_duplicates()
df.to_csv('../../data/drugbank/kegg_drug.csv', index=False)

In [60]:
patway_mapping = pd.read_csv('../../data/vocab/kegg_reactome.csv')
patway_mapping.head()
    

,Source Resource,Source ID,Source Name,Mapping Type,Target Resource,Target ID,Target Name
0,reactome,R-HSA-2978092,Abnormal conversion of 2-oxoglutarate to 2-hyd...,isPartOf,kegg.pathway,path:hsa01210,2-Oxocarboxylic acid metabolism - Homo sapiens...
1,reactome,R-HSA-71406,Pyruvate metabolism and Citric Acid (TCA) cycle,isPartOf,kegg.pathway,path:hsa01210,2-Oxocarboxylic acid metabolism - Homo sapiens...
2,kegg.pathway,path:hsa02010,ABC transporters - Homo sapiens (human),equivalentTo,reactome,R-HSA-1369007,Mitochondrial ABC transporters
3,reactome,R-HSA-5683177,Defective ABCC8 can cause hypoglycemias and hy...,isPartOf,kegg.pathway,path:hsa02010,ABC transporters - Homo sapiens (human)
4,reactome,R-HSA-5679001,Defective ABCC2 causes Dubin-Johnson syndrome,isPartOf,kegg.pathway,path:hsa02010,ABC transporters - Homo sapiens (human)


In [61]:
patway_mapping['Mapping Type'].unique()

array(['isPartOf', 'equivalentTo'], dtype=object)

In [62]:
df = pd.read_csv('../../data/kegg/kegg_drug.csv', low_memory=False)

data = []

for idx, row in df.iterrows():
    drugbank_id = row['drugbank_id']
    kegg_id = row['kegg_id']
    text = row['data']
    pathways = get_kegg_pathway(text)
    pathways_reactome = []
    for pathway in pathways:
        kegg_source = f'(`Source Resource` == "kegg.pathway" and `Source ID` == "path:{pathway}")'
        kegg_target = f'(`Target Resource` == "kegg.pathway" and `Target ID` == "path:{pathway}")'
        mapping_type_part = f'(`Mapping Type` == "isPartOf")'
        mapping_type_equivalent = f'(`Mapping Type` == "isPartOf")'
        if not (res1 := patway_mapping.query(f'{kegg_source} and {mapping_type_part}')).empty:
            for _, row in res1.iterrows():
                pathways_reactome.append({
                    'id':row['Target ID'],
                    'name':row['Target Name']
                })
        elif not (res2 := patway_mapping.query(f'{kegg_source} and {mapping_type_equivalent}')).empty:
            pathways_reactome.append({
                'id':row['Target ID'],
                'name':row['Target Name']
            })
        elif not (res3 := patway_mapping.query(f'{kegg_target} and {mapping_type_part}')).empty:
            for _, row in res3.iterrows():
                pathways_reactome.append({
                    'id':row['Source ID'],
                    'name':row['Source Name']
                })
        elif not (res4 := patway_mapping.query(f'{kegg_target} and {mapping_type_equivalent}')).empty:
            pathways_reactome.append({
                'id':row['Source ID'],
                'name':row['Source Name']
            })
    for pathway in pathways_reactome:
        data.append({
            'kegg_id': kegg_id,
            'drugbank_id': drugbank_id,
            'reactome_id': pathway['id'],
            'reactome_name': pathway['name']
        })

df = pd.DataFrame(data)
df = df.drop_duplicates()
df = df.dropna()
df.to_csv('../../data/kegg/kegg_drug_pathway.csv', index=False)
df

,kegg_id,drugbank_id,reactome_id,reactome_name
0,D09385,DB13083,R-HSA-211916,Vitamins
1,D09385,DB13083,R-HSA-1430728,Metabolism
2,D11464,DB13087,R-HSA-211916,Vitamins
3,D11464,DB13087,R-HSA-1430728,Metabolism
4,D06632,DB13094,R-HSA-211916,Vitamins
...,...,...,...,...
7055,D00471,DB01058,R-HSA-1430728,Metabolism
7056,D00210,DB01059,R-HSA-211916,Vitamins
7057,D00210,DB01059,R-HSA-1430728,Metabolism
7058,D11679,DB15694,R-HSA-211916,Vitamins


In [67]:
df = pd.read_csv('../../data/kegg/kegg_drug_pathway.csv', low_memory=False).drop('kegg_id', axis=1)
df

,drugbank_id,reactome_id,reactome_name
0,DB13083,R-HSA-211916,Vitamins
1,DB13083,R-HSA-1430728,Metabolism
2,DB13087,R-HSA-211916,Vitamins
3,DB13087,R-HSA-1430728,Metabolism
4,DB13094,R-HSA-211916,Vitamins
...,...,...,...
7055,DB01058,R-HSA-1430728,Metabolism
7056,DB01059,R-HSA-211916,Vitamins
7057,DB01059,R-HSA-1430728,Metabolism
7058,DB15694,R-HSA-211916,Vitamins
